In [ ]:
code = 'SBS_SI_PSL'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output\\'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def SBS_per_minute_mtm(bt, start_time, end_time, method, sell_sl, sell_trail, sell_sl_trail, sell_track_original, buy_flag, buy_sl, buy_trail, buy_sl_trail, buy_track_original, sell2_flag, om, std_indicator, indicator_buffer, buffer_minute, seperate=False):
    try:
        start_dt = datetime.datetime.combine(bt.current_date, start_time)
        end_dt = datetime.datetime.combine(bt.current_date, end_time)
        
        # checking straddle indicator
        std_indicator_flag, std_indicator_time = bt.straddle_indicator(start_dt, end_dt, std_indicator, indicator_buffer, buffer_minute)
        
        if std_indicator_flag:
            start_dt = std_indicator_time
        else:
            return None

        ce_scrip, pe_scrip, ce_price, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om)
        if ce_scrip is None: return None

        from_candle_close = True if method == 'CC' else False
        entry_time = start_dt

        #### CE ####
        if sell_sl_trail:
            ce_sl_time, ce_mtm_data0 = bt.sl_check_single_leg_with_sl_trail(start_dt, end_dt, ce_scrip, trail=sell_trail, sl_trail=sell_sl_trail, sl=sell_sl, orderside='SELL', from_candle_close=from_candle_close, per_minute_mtm=True)
        else:
            ce_sl_time, ce_mtm_data0 = bt.sl_check_single_leg(start_dt, end_dt, ce_scrip, sl=sell_sl, orderside='SELL', from_candle_close=from_candle_close, per_minute_mtm=True)

        if sell_track_original:
            _, _, _, _, ce_sl_time, _ = bt.sl_check_single_leg(start_dt, end_dt, ce_scrip, sl=sell_sl, orderside='SELL', from_candle_close=from_candle_close)
            
        #### PE ####
        if sell_sl_trail:
            pe_sl_time, pe_mtm_data0 = bt.sl_check_single_leg_with_sl_trail(start_dt, end_dt, pe_scrip, trail=sell_trail, sl_trail=sell_sl_trail, sl=sell_sl, orderside='SELL', from_candle_close=from_candle_close, per_minute_mtm=True)
        else:
            pe_sl_time, pe_mtm_data0 = bt.sl_check_single_leg(start_dt, end_dt, pe_scrip, sl=sell_sl, orderside='SELL', from_candle_close=from_candle_close, per_minute_mtm=True)

        if sell_track_original:
            _, _, _, _, pe_sl_time, _ = bt.sl_check_single_leg(start_dt, end_dt, pe_scrip, sl=sell_sl, orderside='SELL', from_candle_close=from_candle_close)
            
        ### CE Buy and Sell ###
        ce_mtm_data1, ce_mtm_data2 = pd.Series(), pd.Series()
        if ce_sl_time and buy_flag:

            start_dt = ce_sl_time
            ce_scrip, ce_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om, only='CE')
            
            if ce_scrip is not None:
                # Buy
                if buy_sl_trail:
                    ce_sl_time, ce_mtm_data1 = bt.sl_check_single_leg_with_sl_trail(start_dt, end_dt, ce_scrip, trail=buy_trail, sl_trail=buy_sl_trail, sl=buy_sl, orderside='BUY', from_candle_close=from_candle_close, per_minute_mtm=True)
                else:
                    ce_sl_time, ce_mtm_data1 = bt.sl_check_single_leg(start_dt, end_dt, ce_scrip, sl=buy_sl, orderside='BUY', from_candle_close=from_candle_close, per_minute_mtm=True)

                if buy_track_original:
                    _, _, _, _, ce_sl_time, _ = bt.sl_check_single_leg(start_dt, end_dt, ce_scrip, sl=buy_sl, orderside='BUY', from_candle_close=from_candle_close)

                if ce_sl_time and sell2_flag:

                    start_dt = ce_sl_time
                    ce_scrip, ce_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om, only='CE')

                    if ce_scrip is not None:
                        if sell_sl_trail:
                            ce_sl_time, ce_mtm_data2 = bt.sl_check_single_leg_with_sl_trail(start_dt, end_dt, ce_scrip, trail=sell_trail, sl_trail=sell_sl_trail, sl=sell_sl, orderside='SELL', from_candle_close=from_candle_close, per_minute_mtm=True)
                        else:
                            ce_sl_time, ce_mtm_data2 = bt.sl_check_single_leg(start_dt, end_dt, ce_scrip, sl=sell_sl, orderside='SELL', from_candle_close=from_candle_close, per_minute_mtm=True)


        ### PE Buy and Sell ###
        pe_mtm_data1, pe_mtm_data2 = pd.Series(), pd.Series()
        if pe_sl_time and buy_flag:

            start_dt = pe_sl_time
            pe_scrip, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om, only='PE')
            
            if pe_scrip is not None:
                # Buy
                if buy_sl_trail:
                    pe_sl_time, pe_mtm_data1 = bt.sl_check_single_leg_with_sl_trail(start_dt, end_dt, pe_scrip, trail=buy_trail, sl_trail=buy_sl_trail, sl=buy_sl, orderside='BUY', from_candle_close=from_candle_close, per_minute_mtm=True)
                else:
                    pe_sl_time, pe_mtm_data1 = bt.sl_check_single_leg(start_dt, end_dt, pe_scrip, sl=buy_sl, orderside='BUY', from_candle_close=from_candle_close, per_minute_mtm=True)

                if buy_track_original:
                    _, _, _, _, pe_sl_time, _ = bt.sl_check_single_leg(start_dt, end_dt, pe_scrip, sl=buy_sl, orderside='BUY', from_candle_close=from_candle_close)

                if pe_sl_time and sell2_flag:

                    start_dt = pe_sl_time
                    pe_scrip, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om, only='PE')
                    
                    if pe_scrip is not None:
                        if sell_sl_trail:
                            pe_sl_time, pe_mtm_data2 = bt.sl_check_single_leg_with_sl_trail(start_dt, end_dt, pe_scrip, trail=sell_trail, sl_trail=sell_sl_trail, sl=sell_sl, orderside='SELL', from_candle_close=from_candle_close, per_minute_mtm=True)
                        else:
                            pe_sl_time, pe_mtm_data2 = bt.sl_check_single_leg(start_dt, end_dt, pe_scrip, sl=sell_sl, orderside='SELL', from_candle_close=from_candle_close, per_minute_mtm=True)
                            
        if seperate:
            return ce_mtm_data0, ce_mtm_data1, ce_mtm_data2, pe_mtm_data0, pe_mtm_data1, pe_mtm_data2
        else:
            time_index = get_pm_time_index(bt.current_date)
            
            ce_mtm_data0 = set_pm_time_index(ce_mtm_data0, time_index)
            pe_mtm_data0 = set_pm_time_index(pe_mtm_data0, time_index)
            
            combine_data = ce_mtm_data0 + pe_mtm_data0
            
            if not ce_mtm_data1.empty:
                ce_mtm_data1 = set_pm_time_index(ce_mtm_data1, time_index)
                combine_data += ce_mtm_data1
                
                if not ce_mtm_data2.empty:
                    ce_mtm_data2 = set_pm_time_index(ce_mtm_data2, time_index)
                    combine_data += ce_mtm_data2
                    
            if not pe_mtm_data1.empty:
                pe_mtm_data1 = set_pm_time_index(pe_mtm_data1, time_index)
                combine_data += pe_mtm_data1
                
                if not pe_mtm_data2.empty:
                    pe_mtm_data2 = set_pm_time_index(pe_mtm_data2, time_index)
                    combine_data += pe_mtm_data2
                
            return combine_data

    except Exception as e:
        print(e, [bt.index, bt.current_date, start_time, end_time, method, sell_sl, sell_trail, sell_sl_trail, sell_track_original, buy_flag, buy_sl, buy_trail, buy_sl_trail, buy_track_original, sell2_flag, om, std_indicator, indicator_buffer, buffer_minute])
        return

In [ ]:
def SBS_PSL(bt, start_time, end_time, last_trade_time, trade_interval, method, sell_sl, sell_trail, sell_sl_trail, sell_track_original, buy_flag, buy_sl, buy_trail, buy_sl_trail, buy_track_original, sell2_flag, om, std_indicator, indicator_buffer, buffer_minute):

    try:
        start_dt = datetime.datetime.combine(bt.current_date, start_time)
        end_dt = datetime.datetime.combine(bt.current_date, end_time)
        last_trade_dt = datetime.datetime.combine(bt.current_date, last_trade_time)

        time_range = pd.date_range(start_dt, last_trade_dt, freq=trade_interval.lower())
        time_index = get_pm_time_index(bt.current_date)
        
        entry_time = start_dt
        per_minute_mtm = set_pm_time_index(pd.Series(), time_index)
        for re_time in time_range:
            re_time = re_time.time()
            per_minute_trade = SBS_per_minute_mtm(bt, re_time, end_time, method, sell_sl, sell_trail, sell_sl_trail, sell_track_original, buy_flag, buy_sl, buy_trail, buy_sl_trail, buy_track_original, sell2_flag, om, std_indicator, indicator_buffer, buffer_minute)
            if per_minute_trade is not None:
                per_minute_mtm += per_minute_trade

        notinal_value = 12
        fund = 100_00_00_00
        total_minutes = len(time_range)
        future_price = bt.future_data['close'].iloc[0]
        margin_per_share = future_price * (notinal_value / 100)
        per_minute_margin = int(fund/total_minutes)
        shares_per_minute = int(per_minute_margin/margin_per_share)
        per_minute_mtm_total = per_minute_mtm * shares_per_minute
        
        mtm_dict = {}
        for mtm_percent in mtm_percent_stop:
            check_mtm = fund * mtm_percent/100

            try:
                if check_mtm > 0:
                    time = per_minute_mtm_total[per_minute_mtm_total > check_mtm].index[0]
                elif check_mtm < 0:
                    time = per_minute_mtm_total[per_minute_mtm_total < check_mtm].index[0]

                mtm = per_minute_mtm_total.loc[time + datetime.timedelta(minutes=1)]
            except:
                time, mtm = '', per_minute_mtm_total.iloc[-1]

            mtm_dict[mtm_percent] = (time, mtm)
            
        mtm_time_list = [item for value in mtm_dict.values() for item in value]
       
        return [code, bt.index, start_time, end_time, last_trade_time, trade_interval, method, sell_sl, sell_trail, sell_sl_trail, sell_track_original, buy_flag, buy_sl, buy_trail, buy_sl_trail, buy_track_original, sell2_flag, om, std_indicator, indicator_buffer, buffer_minute, bt.current_date.date(), bt.current_date.day_name(), bt.dte, entry_time.time(), future_price, fund] + mtm_time_list
    except Exception as e:
        print(e, [bt.index, bt.current_date, start_time, end_time, last_trade_time, trade_interval, method, sell_sl, sell_trail, sell_sl_trail, sell_track_original, buy_flag, buy_sl, buy_trail, buy_sl_trail, buy_track_original, sell2_flag, om, std_indicator, indicator_buffer, buffer_minute])
        return

In [ ]:
for row_idx in range(len(meta_data)):

    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, dte, from_date, to_date, start_time, end_time, date_lists = get_meta_row_data(meta_row, pickle_path)
            max_re = 7

            mtm_percent_stop = [-0.1, -0.2, -0.3, -0.4, -0.5, -0.6, -0.7, -0.8, -0.9, -1, -1.25, -1.5, -1.75, -2, -2.5, -3, -3.5, -4, -10, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 1.25, 1.5, 1.75, 2, 2.5, 3, 3.5, 4]
            log_cols = ('P_Strategy/P_Index/P_StartTime/P_EndTime/P_LastTradeTime/P_TradeInterval/P_Method/P_SellSL/P_SellTrail/P_SellSLTrial/P_SellTrackOriginal/P_BuyFlag/P_BuySL/P_BuyTrail/P_BuySLTrial/P_BuyTrackOriginal/P_Sell2Flag/P_OM/P_StdIndicator/P_IndicatorBuffer/P_BufferMinute/Date/Day/DTE/Entry.Time/Future/Fund')
            for mtmp in mtm_percent_stop:
                log_cols += f'/{mtmp:.2f}.Time/{mtmp:.2f}.PNL'
            log_cols = log_cols.split('/')

            for current_date in date_lists:

                file_name = f"{index} {current_date.date()} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")
                    
                    bt = IntradayBacktest(pickle_path, index, current_date, dte, start_time, end_time)
                    
                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)
                        
                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [SBS_PSL(bt, row['entry_time'], row['exit_time'], row['last_trade_time'], row['trade_interval'], row['method'], row['sell_sl'], row['sell_trail'], row['sell_sl_trail'], row['sell_track_original'], row['buy_flag'], row['buy_sl'], row['buy_trail'], row['buy_sl_trail'], row['buy_track_original'], row['sell2_flag'], row['om'], row['std_indicator'], row['indicator_buffer'], row['buffer_minute']) for idx, row in tqdm(chunk_parameter.iterrows(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)
                        
                        del chunk
                        del chunk_parameter
                        gc.collect()

                    del bt
                    gc.collect()
                    
                    t2 = datetime.datetime.now()
                    print(t2-t1)
        except Exception as e:
            input(str(e))